# Automating a Web Pen-Test on Real Targets: Chains → RAG → Agents + MCP
### A Concrete, Real-Data Walkthrough of the Three Lectures

**Audience:** Graduate students who have completed the LangChain, RAG, and MCP modules.

**Goal:** Re-run the same three-stage story as `cybersecurity_automation.ipynb`, but on **real, authorized vulnerable websites** instead of a simulated `nmap`. You will see genuine HTTP recon data flow through each stage:

1. 🔗 **Chains** — `web recon → LLM parses → LLM analyzes → report`. *The LLM guesses, with no OWASP references.*
2. 🔍 **RAG** — `… → retrieve web-security guidance → analyze → report`. *Answers grounded in a real knowledge base, citing OWASP categories.*
3. 🤖 **Agent + MCP** — *the agent decides which recon tool to call*, through a standard `web_recon_server.py`.

> ⚠️ **Ethics & authorization.** Recon here targets only **authorized** sites: **`scanme.nmap.org`** (published by the Nmap Project expressly so people can test scanning tools) and the **Acunetix `vulnweb.com` family** (`testphp.vulnweb.com`, …). Every call makes at most **one ordinary HTTP GET** — no scanning, fuzzing, or exploitation. **Never point these tools at a site you are not explicitly authorized to test.**

> **Reproducibility.** If the network is blocked, recon falls back to a clearly-labelled cached sample, so every cell still runs. Stages 1–2 and the offline agent need **no API key**; the **live agent** needs `OPENAI_API_KEY` and the `mcp` SDK.

> **This vs. the VM project.** This notebook is the *worked example* over web targets. Your **project** is to stand up a local exploitable VM (e.g., Metasploitable), run `nmap` and other tools against it, and adapt these three stages — see the closing cell.

## 🛠️ Setup

In [ ]:
# If needed:
# !pip install -q langchain-core sentence-transformers
# Optional (Stage 3 live agent): !pip install -q mcp langchain-mcp-adapters langgraph langchain-openai

import re, ssl, urllib.request
from typing import List
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage

print("Core imports ready.")

### Real (but safe) web recon, with an offline cache

We only contact **authorized** targets, we make a single GET per call, and if the network is unavailable we fall back to a **cached sample** so the lecture is reproducible. This is the same recon logic the `web_recon_server.py` exposes as MCP tools in Stage 3.

In [ ]:
ALLOWED_HOSTS = {"scanme.nmap.org", "testphp.vulnweb.com", "testasp.vulnweb.com",
                 "testaspnet.vulnweb.com", "testhtml5.vulnweb.com"}

SECURITY_HEADERS = ["Strict-Transport-Security", "Content-Security-Policy",
                    "X-Frame-Options", "X-Content-Type-Options", "Referrer-Policy"]

# Representative cached responses (publicly-known traits of these authorized test
# sites). Used only when the live fetch fails.
CACHED = {
    "http://scanme.nmap.org": {"status": 200, "headers": {
        "Server": "Apache/2.4.7 (Ubuntu)", "Content-Type": "text/html",
        "Connection": "close"}},
    "http://testphp.vulnweb.com": {"status": 200, "headers": {
        "Server": "nginx/1.19.0", "Content-Type": "text/html; charset=UTF-8",
        "Connection": "close", "X-Powered-By": "PHP/5.6.40"}},
    "http://testasp.vulnweb.com": {"status": 200, "headers": {
        "Server": "Microsoft-IIS/8.5", "Content-Type": "text/html",
        "X-Powered-By": "ASP.NET", "X-AspNet-Version": "4.0.30319"}},
}

def host_of(url): return url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0]

def http_recon(url, timeout=8):
    """One ordinary GET against an AUTHORIZED url. Falls back to cache if offline."""
    if host_of(url) not in ALLOWED_HOSTS:
        raise ValueError(f"Refused: {host_of(url)} is not authorized.")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "edu-web-recon/1.0"})
        with urllib.request.urlopen(req, timeout=timeout,
                                    context=ssl.create_default_context()) as r:
            return {"status": r.status, "headers": dict(r.headers), "live": True, "url": url}
    except Exception as e:
        c = CACHED.get(url, {"status": None, "headers": {}})
        return {"status": c["status"], "headers": dict(c["headers"]),
                "live": False, "url": url, "error": str(e)}

# scanme.nmap.org is authorized by the Nmap Project for testing. To use another
# allowed site instead, set TARGET to e.g. "http://testphp.vulnweb.com".
TARGET = "http://scanme.nmap.org"
recon = http_recon(TARGET)
print(f"Recon of {TARGET}  ({'LIVE' if recon['live'] else 'CACHED'}):")
print("  HTTP", recon["status"])
for k, v in recon["headers"].items():
    print(f"  {k}: {v}")

Turn the raw response into a list of **findings** — the web-recon analog of nmap's service list. Each finding is a missing security header, a version disclosure, or cleartext HTTP.

In [ ]:
def analyze_findings(recon: dict) -> List[str]:
    headers = recon["headers"]
    present = {k.lower() for k in headers}
    findings = []
    if recon["url"].lower().startswith("http://"):
        findings.append("Cleartext HTTP (no TLS)")
    for h in SECURITY_HEADERS:
        if h.lower() not in present:
            findings.append(f"Missing security header: {h}")
    if "server" in present:
        findings.append(f"Server version disclosure: {headers.get('Server')}")
    xpb = headers.get("X-Powered-By")
    if xpb:
        eol = " (PHP 5.x is end of life)" if "php/5" in xpb.lower() else ""
        findings.append(f"Technology disclosure: {xpb}{eol}")
    return findings

findings = analyze_findings(recon)
print("Findings:")
for f in findings:
    print("  -", f)

---
## 🔗 STAGE 1 — Chains: An LLM-Powered Web-Recon Assistant

*Theme: “Use LangChain to turn raw recon into a readable report.”*

# 🔗 Stage 1 — Recon → Parse → Analyze → Report (a Chain)

- Fetch real HTTP headers from an authorized target
- Extract a list of **findings** (missing headers, disclosures)
- LLM **analyzes** each finding for likely risk
- LLM **generates** a readable report
- Wired as a LangChain chain: `prompt | model | parser`

> ### 🎤 Instructor Script
>
> We start exactly where the simulated capstone did, but with real data. We fetch the actual HTTP headers of an authorized vulnerable site, turn them into a list of findings, and feed those to an LLM acting as a security analyst, all strung together as a LangChain chain. The shape is the familiar prompt-model-parser you learned in the LangChain module, now applied to genuine recon output. We use the offline stand-in model so it runs in class; the live version is a one-line swap shown later. Notice this is already useful — in minutes you have an assistant that turns a raw header dump into prose.

In [ ]:
def _analyst_from_memory(prompt_value):
    """Stand-in LLM playing 'analyst from general knowledge' (it guesses)."""
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    items = [l.strip("- ").strip() for l in text.splitlines() if l.strip().startswith("-")]
    body = ("Based on general knowledge, these findings *may* indicate weaknesses "
            "(UNVERIFIED — the model is guessing, with no references):\n")
    for f in items:
        body += f"  - {f}: probably worth hardening; could enable some attack.\n"
    return AIMessage(content=body + "Recommend manual verification.")

analyst_memory_llm = RunnableLambda(_analyst_from_memory)

report_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a web-application security assistant. Analyze the findings and write a "
               "short report."),
    ("user", "Findings:\n{findings}"),
])

stage1_chain = report_prompt | analyst_memory_llm | StrOutputParser()

report = stage1_chain.invoke({"findings": "\n".join(f"- {f}" for f in findings)})
print("=== STAGE 1 REPORT (chain, LLM from memory) ===\n")
print(report)

# ⚠️ Stage 1 — The Critical Limitation

- The LLM **guesses** from training memory
- No OWASP category, no CWE, no concrete remediation
- Cannot tie a finding to an authoritative standard
- → **hallucination risk** in a security context = dangerous
- This motivates Stage 2 (RAG)

> ### 🎤 Instructor Script
>
> Now the teaching moment. Re-read that report: it is fluent and plausible, but it cites nothing. The model is guessing from general training, so it gives no OWASP category, no CWE identifier, and no standards-based remediation a real assessment needs. In security, a vague or wrong answer is dangerous — a hallucinated 'looks fine' can leave a real hole open. The lesson is that an LLM's memory is the wrong source of truth for findings that should map to authoritative guidance. That is exactly what Retrieval-Augmented Generation fixes, which is Stage 2.

In [ ]:
print("What Stage 1 produced:")
print("  - source of 'facts' : the LLM's training memory (a guess)")
print("  - OWASP / CWE refs   : none")
print("  - remediation        : vague")
print("  - trustworthy        : NOT for a real assessment")
print("\nFix: ground each finding in a real web-security knowledge base -> Stage 2 (RAG).")

---
## 🔍 STAGE 2 — RAG: Ground Each Finding in OWASP Guidance

*Theme: “From guessing → retrieving real security guidance.”*

# 🔍 Stage 2 — Retrieve Real Guidance, Then Analyze

- Embed `web_security_kb.md` into a vector store (the RAG module)
- For each finding, **retrieve** the matching OWASP guidance
- Feed the *retrieved* guidance to the LLM as context
- Report now cites **OWASP categories** and concrete fixes
- Pipeline: `recon → findings → retrieve → analyze → report`

> ### 🎤 Instructor Script
>
> Stage 2 upgrades the assistant exactly as the RAG lecture promised: we replace the model's memory with retrieval from a real knowledge base. Using the embeddings and vector-store tools from the RAG module, we index `web_security_kb.md`, then for each finding we retrieve the most relevant guidance and pass it into the prompt as context. The model's job changes from recall to reading comprehension over supplied facts. The report now references concrete OWASP categories like A05 Security Misconfiguration with real remediation. Same recon, same model, but grounded output.

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

class STEmbeddings(Embeddings):
    def __init__(self, name="all-MiniLM-L6-v2"):
        self.m = SentenceTransformer(name)
    def embed_documents(self, texts): return self.m.encode(texts, normalize_embeddings=True).tolist()
    def embed_query(self, text):      return self.m.encode(text, normalize_embeddings=True).tolist()

kb_text = open("web_security_kb.md", encoding="utf8").read()
sections = [s.strip() for s in kb_text.split("#") if s.strip()]
docs = [Document(page_content=s, metadata={"title": s.splitlines()[0]}) for s in sections]

kb_store = InMemoryVectorStore(STEmbeddings())
kb_store.add_documents(docs)
print(f"Indexed {len(docs)} guidance entries into the vector store.\n")

def retrieve_guidance(finding: str, k: int = 1, threshold: float = 0.30) -> str:
    hits = kb_store.similarity_search_with_score(finding, k=k)
    kept = [d.page_content for d, score in hits if score >= threshold]
    return kept[0] if kept else f"No guidance found for: {finding}"

for f in findings:
    print(f"  {f[:42]:44s} ->  {retrieve_guidance(f).splitlines()[0]}")

Now build the **RAG chain**: retrieve guidance for the findings, then have the (stand-in) LLM write a grounded report that cites the OWASP categories from the retrieved context.

In [ ]:
def _analyst_grounded(prompt_value):
    """Stand-in LLM that answers ONLY from the retrieved guidance (extractive)."""
    text = prompt_value.to_string() if hasattr(prompt_value, "to_string") else str(prompt_value)
    context = text.split("# Retrieved guidance", 1)[-1].split("Human:", 1)[0].strip()
    owasp = sorted(set(re.findall(r"A0\d:2021[^.\n]*", context)))
    head = f"GROUNDED REPORT — cites {len(owasp)} OWASP categor(ies): {'; '.join(owasp)}\n\n"
    return AIMessage(content=head + context[:700] + (" ..." if len(context) > 700 else ""))

grounded_llm = RunnableLambda(_analyst_grounded)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a web-app security assistant. Use ONLY the retrieved guidance below.\n"
               "# Retrieved guidance\n{context}"),
    ("user", "Assess {url} given these findings:\n{findings}"),
])

def build_context(_):
    return "\n\n".join(retrieve_guidance(f) for f in findings)

stage2_chain = (
    {"context": RunnableLambda(build_context),
     "findings": lambda _: "\n".join(f"- {f}" for f in findings),
     "url": RunnableLambda(lambda u: u)}
    | rag_prompt | grounded_llm | StrOutputParser()
)

print("=== STAGE 2 REPORT (RAG, grounded in OWASP guidance) ===\n")
print(stage2_chain.invoke(TARGET))

# ✅ Stage 2 — What Improved

- Findings mapped to **real OWASP categories**, not memory
- Report gives **concrete, standards-based** remediation
- Hallucination sharply **reduced**
- Knowledge base is **updatable** without retraining
- Still a *fixed* pipeline — Stage 3 makes it dynamic

> ### 🎤 Instructor Script
>
> Compare the two reports and the win is obvious: Stage 2 maps each finding to a real OWASP category with standards-based remediation drawn from documents, while Stage 1 offered fluent guesses. Accuracy is up, hallucination is down, and the knowledge base can be refreshed the moment guidance changes — no retraining. But notice what is still true: the pipeline is fixed. We hard-coded recon, then findings, then retrieve, then analyze. What if a finding should trigger a deeper look, or fetching robots.txt? For that the system must decide for itself — Stage 3: agents and MCP.

In [ ]:
print(f"{'':22s}{'STAGE 1 (chain)':22s}{'STAGE 2 (RAG)'}")
print("-" * 62)
for label, a, b in [("Source of facts",  "LLM memory", "web_security_kb.md"),
                    ("OWASP references", "none",       "yes (e.g. A05:2021)"),
                    ("Remediation",      "vague",      "concrete, standards-based"),
                    ("Hallucination",    "high",       "reduced"),
                    ("Updatable",        "retrain",    "edit the KB")]:
    print(f"{label:22s}{a:22s}{b}")

---
## 🤖 STAGE 3 — Agents + MCP: Let the System Decide

*Theme: “From pipelines → autonomous web-recon systems.”*

# 🤖 Stage 3 — An Agent Orchestrates MCP Recon Tools

- `web_recon_server.py` exposes `fetch_headers`, `check_security_headers`, `get_robots`, `lookup_guidance`
- An **agent** decides *which* tool to call, *when*, and *with what*
- Loop: reason → call a tool → observe → repeat → report
- Tools are standardized & reusable (swap/add servers freely)
- Final pipeline is **dynamic**, not hard-coded

> ### 🎤 Instructor Script
>
> Stage 3 removes the last constraint: the fixed sequence. Instead of us scripting recon-then-retrieve, we expose those capabilities as MCP tools on `web_recon_server.py` and hand them to an agent. The agent runs the reason-act-observe loop: it decides to fetch the headers, reads them, decides to check which security headers are missing, looks up guidance for each, and writes the report. The control flow now comes from the model, and because the tools are MCP-standardized we could add a TLS-checker or a CVE server later without touching the agent. First a deterministic offline version with no key; then the live LangGraph + MCP agent.

In [ ]:
# --- Offline agent: a deterministic loop that DECIDES its next tool call ------
def tool_fetch_headers(url): return http_recon(url)
def tool_check_security_headers(url): return analyze_findings(http_recon(url))
def tool_lookup_guidance(finding): return retrieve_guidance(finding).splitlines()[0]
AGENT_TOOLS = {"fetch_headers": tool_fetch_headers,
               "check_security_headers": tool_check_security_headers,
               "lookup_guidance": tool_lookup_guidance}

def offline_agent(goal_url, max_steps=10):
    print(f"GOAL: assess {goal_url}\n")
    fetched = False; findings_left = []; report = []
    for step in range(1, max_steps + 1):
        # --- the 'policy' (a real agent lets the LLM choose this) ---
        if not fetched:
            action, args = "check_security_headers", {"url": goal_url}
        elif findings_left:
            action, args = "lookup_guidance", {"finding": findings_left.pop(0)}
        else:
            print(f"  step {step}: REASON -> enough info, write the report"); break
        print(f"  step {step}: REASON -> call {action}({args})")
        result = AGENT_TOOLS[action](**args)
        shown = result if isinstance(result, str) else "; ".join(result)
        print(f"           OBSERVE -> {shown[:72]}{'...' if len(shown) > 72 else ''}")
        if action == "check_security_headers":
            fetched = True; findings_left = list(result)
        else:
            report.append(result)
    print("\n=== STAGE 3 REPORT (agent-orchestrated) ===")
    for r in report:
        print("  -", r)

offline_agent(TARGET)

# 🚀 Stage 3 (Live) — LangGraph Agent over the MCP Server

- `load_mcp_tools` → MCP tools become LangChain tools
- `create_react_agent(llm, tools)` builds the agent loop for you
- The LLM picks recon tools dynamically at runtime
- Needs `OPENAI_API_KEY` + the `mcp` SDK (else this step skips)
- Connects to `web_recon_server.py` over stdio

> ### 🎤 Instructor Script
>
> Finally, the real thing. We connect to `web_recon_server.py` over stdio, load its tools as LangChain tools with `load_mcp_tools`, and build a LangGraph ReAct agent with `create_react_agent`, backed by a real OpenAI model. Now the agent genuinely decides: it calls check_security_headers on the authorized target, looks up guidance for each finding, and synthesizes a grounded report — and it would just as easily use a new tool if we added one to the server. This step needs an API key and the `mcp` package, so it is guarded; without them it prints how to run it. Everything before this ran offline, so the lecture works either way.

In [ ]:
import os, sys

async def run_live_agent(goal):
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client
    from langchain_mcp_adapters.tools import load_mcp_tools
    from langgraph.prebuilt import create_react_agent
    from langchain_openai import ChatOpenAI
    import asyncio

    # errlog -> real file: passing Jupyter's sys.stderr triggers a fileno error;
    # swallowing the server's stderr would turn a dead server into a silent hang.
    errlog = open("web_recon_server_stderr.log", "w")
    params = StdioServerParameters(command=sys.executable, args=["web_recon_server.py"])
    async with stdio_client(params, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await asyncio.wait_for(session.initialize(), timeout=30)   # fail loud, don't hang
            tools = await load_mcp_tools(session)                      # MCP -> LangChain tools
            agent = create_react_agent(ChatOpenAI(model="gpt-4o-mini", temperature=0), tools)
            result = await asyncio.wait_for(
                agent.ainvoke({"messages": [("user", goal)]}), timeout=120)
            return result["messages"][-1].content

def have(mod):
    import importlib.util
    return importlib.util.find_spec(mod) is not None

def run_async(coro):
    """Run a coroutine from anywhere, including a Jupyter cell.

    A Jupyter cell already runs inside an event loop, and asyncio.run() refuses a
    nested one. nest_asyncio is NOT used: on Windows, spawning the stdio server
    subprocess needs a ProactorEventLoop that a re-entrant loop can't drive. So we
    run the coroutine in a SEPARATE THREAD with its own fresh ProactorEventLoop.
    """
    import asyncio, threading
    try:
        asyncio.get_running_loop(); in_loop = True
    except RuntimeError:
        in_loop = False
    if not in_loop:
        return asyncio.run(coro)
    box = {}
    def worker():
        loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
        try:
            box["value"] = loop.run_until_complete(coro)
        except BaseException as exc:
            box["error"] = exc
        finally:
            loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]

if os.getenv("OPENAI_API_KEY") and have("mcp") and have("langgraph") and have("langchain_openai"):
    goal = (f"Assess {TARGET}: check its security headers, look up OWASP guidance for each finding, "
            "and give me a short risk report. Only use the provided tools.")
    print(run_async(run_live_agent(goal)))
else:
    print("Live agent skipped (needs OPENAI_API_KEY + mcp + langgraph + langchain-openai).")
    print("Set the key and `python web_recon_server.py` is launched automatically over stdio.")
    print("\nThe agent would: check_security_headers(TARGET) -> lookup_guidance(each finding)")
    print("-> grounded report, choosing each tool call itself instead of following a fixed script.")

# 🔥 The Three Lectures in One Table

- Stage 1 Chains: orchestration ✅, data = LLM memory, fixed
- Stage 2 RAG: + real OWASP guidance, grounded, still fixed
- Stage 3 Agent+MCP: dynamic decisions, standardized recon tools
- Accuracy: medium → high → high
- Automation: low → medium → high

> ### 🎤 Instructor Script
>
> Step back and see the arc whole, now on real data. Stage 1 gave us orchestration with chains, but facts came from the model's memory and the pipeline was fixed. Stage 2 kept the structure but grounded the findings in real OWASP guidance through RAG, trading guesses for citations. Stage 3 broke the fixed pipeline open: an agent decides the steps at runtime, and MCP gives it a standardized, swappable recon toolbox. The one-line story to leave students with: we start by using LLMs to interpret recon, then make them accurate with real guidance, and finally make them autonomous with agents and standardized tools.

In [ ]:
rows = [
    ("Orchestration",    "Chains",     "Chains",            "Agents"),
    ("Data source",      "LLM memory", "RAG (OWASP KB)",    "RAG + MCP tools"),
    ("Accuracy",         "medium",     "high",              "high"),
    ("Automation",       "low",        "medium",            "high"),
    ("Flexibility",      "fixed",      "fixed",             "dynamic"),
    ("Tool integration", "manual",     "manual+retrieval",  "autonomous (agent+MCP)"),
]
print(f"  {'CAPABILITY':18s}{'LECTURE 1':16s}{'LECTURE 2':20s}{'LECTURE 3'}")
print("  " + "-" * 72)
for r in rows:
    print(f"  {r[0]:18s}{r[1]:16s}{r[2]:20s}{r[3]}")
print("\n  Story: interpret (chains) -> accurate (RAG) -> autonomous (agents + MCP).")

---
## 🎯 Your Project — Take This to a Local Exploitable VM

This notebook used **non-intrusive HTTP recon** on authorized public targets so it is safe to run anywhere. Your project goes deeper, in an environment you fully control:

1. **Stand up a lab.** Install an intentionally-vulnerable VM (e.g., **Metasploitable 2/3** or OWASP **Juice Shop**) on an **isolated, host-only network**.
2. **Real scanning.** Run `nmap -sV` and other tools (e.g., `nikto`, `whatweb`, `gobuster`) against the VM — now you have *service* and *web* findings from a real scanner.
3. **Adapt the three stages.** Feed that real scan output through Stage 1 (chain report), Stage 2 (RAG over a CVE / OWASP knowledge base), and Stage 3 (an MCP server that wraps your tools so an agent can orchestrate them).
4. **Compare & reflect.** Show how grounding and autonomy change the output, and document the limitations you hit.

> ⚠️ **Rules of engagement.** Run scanners and exploits **only** against your own isolated lab VM or a target you are explicitly authorized to test. Keep the VM off the public network. Keep API keys in `.env` / Colab Secrets — never in code.